# 77 — Train BGE-M3 bi-encoder v2 (Stage A, text-only P0/P1)

Improved recall encoder with the Phase-0 fixes baked into the scripts on branch `recall-union-lgbm`:
- track-text parity (catalog embedded in the id_to_metadata format the positives use)
- serve-safe `[STATE]` intent block (StateTracker user_state; NO leaky `thought`)
- clean negatives (global-gold exclusion) + simans (no-drop of the hard new-artist tail)
- best-checkpoint-on-full-catalog-nDCG (deploy best, not last epoch)

This is the disciplined v1 bundle: TEXT-ONLY bge-m3, distillation/multipositive OFF. Measure in nb78 vs the +0.0387 baseline before adding multimodal/cross-encoder.

Run order: 1 setup -> 2 config -> 3 precompute state (optional) -> 4 build triples -> 5 train -> 6 embed catalog. Then run nb78 to evaluate.


In [ ]:
# 1) Setup — clone branch + HF auth + Drive + deps.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# Stop transformers from importing Flax/TF (which start JAX threads). The train
# DataLoader uses num_workers=2 -> fork(); JAX threads + fork = deadlock (no logs).
# These env vars are inherited by the `!python` subprocesses in cells 4-6.
os.environ['USE_FLAX'] = '0'
os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# bm25s is imported transitively by mcrs.retrieval_modules (the builder needs it);
# FlagEmbedding loads the bge-m3 miner. Both are required by cell 4.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'sentence-transformers>=3.0' 'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' \
    'tensorboard' 'bm25s' 'FlagEmbedding>=1.3'
# Colab ships torchao 0.10.0; PEFT's LoRA dispatch hard-raises (wants >0.16) even
# though we use plain LoRA, not torchao quantization. Remove it so get_peft_model works.
!pip uninstall -y torchao
print('setup done')

In [ ]:
# 2) CONFIG — v2 (P0/P1) bundle.
HUB_USER  = 'OrRim123'
RUN_NAME  = 'bge-base-en-music-v2'     # backbone in the name; NEW repo (protects bge-m3 artifacts)
BGE_MODEL = 'BAAI/bge-base-en-v1.5'    # ~109M, ~5x faster than bge-m3 for iteration; English-only.
                                       # Final judged run may prefer multilingual bge-m3 — compare in nb78.

# Data scope
MAX_INPUT_ROWS = 20000                 # 20000 = ~1h smoke (enough to see the trend);
                                       # set 0 for the full ~121K (final/judged model).
SKIP_MINING    = False                 # build fresh triples with the P0 fixes

# HN mining (simans + global-gold exclusion are the script defaults now)
MINING_STRATEGY = 'simans'
POOL_SIZE       = 1000
N_NEGATIVES     = 15
MINING_BATCH    = 128

# [STATE] intent block (serve-safe). OFF by default: it needs the heavy
# StateTracker precompute (cell 3) and is a separate gated increment. Turn ON
# (and set nb78 USE_STATE=True) only for a controlled 121k+STATE run.
USE_STATE   = False
STATE_CACHE = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache'   # {..}/state/{sess}__{turn}.json

# Training recipe (tuned; best-ckpt-on-nDCG is automatic). LoRA targets
# (query/key/value/dense) are BERT/XLM-R common, so r64/a128 work on bge-base too
# (~10M trainable, ~9-10% — drop to r32 only if it overfits).
EPOCHS = 3; LR = 2e-5; TEMPERATURE = 0.02; LORA_RANK = 64; LORA_ALPHA = 128
PER_DEVICE_BATCH = 16; GRAD_ACCUM = 16         # eff batch 256; in-batch negatives 256/query
# VAL_FULL_EVERY=50 fits the short ~211-step smoke (~4 full-catalog points to read
# the trend). RAISE back to ~200 for the full 121k run (val over 47k×12k every 50
# steps is far too much eval overhead on a ~1283-step run).
VAL_FULL_EVERY = 50; SPLIT_KEY = 'user_id'; VAL_FRACTION = 0.10

# Derived
HUB_REPO        = f'{HUB_USER}/recsys2026-{RUN_NAME}'
SAFE            = RUN_NAME.replace('-', '_')
TRIPLES         = f'experiments/cache/retrieval_v2/triples_{SAFE}.jsonl'
TRAIN_OUT       = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/training/{SAFE}'
DRIVE_CACHE     = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local'
import os; os.makedirs(os.path.dirname(f'/content/recsys2026/{TRIPLES}'), exist_ok=True)
print('config:', dict(RUN_NAME=RUN_NAME, BGE_MODEL=BGE_MODEL, MINING_STRATEGY=MINING_STRATEGY,
      USE_STATE=USE_STATE, MAX_INPUT_ROWS=MAX_INPUT_ROWS, PER_DEVICE_BATCH=PER_DEVICE_BATCH,
      EPOCHS=EPOCHS, HUB_REPO=HUB_REPO))

In [ ]:
# 3) (optional, GPU) Precompute StateTracker user_state over the train split for
#    the [STATE] block. Idempotent (caches per turn). ~the slow step; skip if
#    USE_STATE=False or the cache already exists on Drive.
if USE_STATE:
    %cd /content/recsys2026
    !python scripts/precompute_train_user_state.py \
        --lm-type Qwen/Qwen2.5-1.5B-Instruct \
        --cache-dir {STATE_CACHE} \
        --conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset --split train \
        --max-sessions {0 if MAX_INPUT_ROWS==0 else 3000}
else:
    print('USE_STATE=False -> skipping state precompute')


In [ ]:
# 4) Build triples (hard-neg mining). simans + global-gold exclusion are defaults.
#    --state-cache-dir adds the [STATE] block (parity with serve). --history-corpus-types
#    = the 3 canonical fields (track_text parity).
%cd /content/recsys2026
import os
if SKIP_MINING and os.path.exists(TRIPLES):
    print('SKIP_MINING -> reusing', TRIPLES)
else:
    state_flag = f'--state-cache-dir {STATE_CACHE}' if USE_STATE else ''
    rows_flag  = f'--max-rows {MAX_INPUT_ROWS}' if MAX_INPUT_ROWS else ''
    !python scripts/build_bi_encoder_training_data.py \
        --train-conv-hf talkpl-ai/TalkPlayData-Challenge-Dataset \
        --track-meta-hf talkpl-ai/TalkPlayData-Challenge-Track-Metadata \
        --bge-m3-model {BGE_MODEL} \
        --query-mode bge_m3_structured \
        --history-corpus-types track_name,artist_name,album_name \
        --mining-strategy {MINING_STRATEGY} --pool-size {POOL_SIZE} --k-negs {N_NEGATIVES} \
        --batch-size {MINING_BATCH} --exclude-global-golds {state_flag} {rows_flag} \
        --output {TRIPLES}
!wc -l {TRIPLES}
!head -c 600 {TRIPLES}


In [ ]:
# 5) Train + merge LoRA -> base + push to Hub. Best-on-full-catalog-nDCG is saved
#    to {TRAIN_OUT}/best and the merge prefers it automatically.
%cd /content/recsys2026
!python scripts/train_bi_encoder.py \
    --triples {TRIPLES} --base-model {BGE_MODEL} \
    --output-dir {TRAIN_OUT} --hub-repo {HUB_REPO} --merge \
    --epochs {EPOCHS} --lr {LR} --temperature {TEMPERATURE} \
    --lora-rank {LORA_RANK} --lora-alpha {LORA_ALPHA} \
    --per-device-batch-size {PER_DEVICE_BATCH} --gradient-accumulation-steps {GRAD_ACCUM} \
    --query-max-len 384 --passage-max-len 192 \
    --in-batch-negs --n-negatives {N_NEGATIVES} \
    --split-key {SPLIT_KEY} --val-fraction {VAL_FRACTION} \
    --val-full-catalog-every-n-steps {VAL_FULL_EVERY} \
    --checkpoint-every-n-epochs 1
print('pushed merged model ->', HUB_REPO + '-merged')


In [ ]:
# 6) Embed the catalog in the CANONICAL id_to_metadata format (track-text parity).
#    DENSE_LOCAL reads this pickle; it MUST match the positives the encoder trained on.
%cd /content/recsys2026
!python scripts/embed_catalog.py \
    --model {HUB_REPO}-merged --label {RUN_NAME}-merged \
    --fields track_name artist_name album_name \
    --doc-format id_to_metadata \
    --cache-root {DRIVE_CACHE} --batch-size 64
print('catalog embedded (id_to_metadata format) for', HUB_REPO + '-merged')


## Next: evaluate in nb78

Open `colab/78_e2e_stageA_stageB_rerank_ndcg.ipynb`, set `BGE_HUB = '{HUB_USER}/recsys2026-{RUN_NAME}-merged'` (it already defaults to the bge-base-en repo) and `USE_STATE` to match this notebook, and run the additivity + nDCG gates against the +0.0387 / 0.1637 baseline.